In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt



: 

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

In [ ]:
df = pd.read_parquet("/Users/riturajbhattacharjee/Desktop/yellow_tripdata_2023-02 (1).parquet")
df['store_and_fwd_flag'] = df['store_and_fwd_flag'].map({'N': 0, 'Y': 1})
df.head()


In [ ]:
#  vendorId:- 1(Uber) , 2(Ola)



In [ ]:
# RatecodeID:- 1.0(Standard rate)


In [ ]:
# store_and_fwd_flag:- indicates if trip was stored first and sent later; 0.0(Sent immediately), 1.0(stored then sent later)

In [ ]:
# payment_type:- 0(Data is missing),1(Credit Card),2(Cash),3(No charge), 4(Dispute)

In [ ]:
# mta_tax:- std. NYC taxi tax; $0.50, $ -0.50(if the ride is cancelled or refunded)

In [ ]:
 df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
#  Fare Amount Distribution (Histogram Plot)


In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(df['fare_amount'], bins=100)

plt.xlim(0, 100)
plt.title("Fare Amount Distribution ")
plt.xlabel("Fare Amount")
plt.ylabel("Count")

plt.show()

In [ ]:
# Violin Plot


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# IQR filtering
Q1 = df['fare_amount'].quantile(0.25)
Q3 = df['fare_amount'].quantile(0.75)
IQR = Q3 - Q1

clean_df = df[(df['fare_amount'] >= Q1 - 1.5*IQR) &
              (df['fare_amount'] <= Q3 + 1.5*IQR)]

# Plot
plt.figure(figsize=(10,5))
sns.violinplot(x=clean_df['fare_amount'], color='orange')

plt.title("Fare Amount Distribution")
plt.xlabel("Fare Amount")
plt.show()

### HeatMap


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,8))

corr = df.corr(numeric_only=True)

sns.heatmap(corr, annot=True, fmt=".1f", cmap="coolwarm")

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
#Trip Distance Distribution (Trip Distance vs No. of trips)

#1e6 -> 1 x 10^6 = 1000000(10 lakhs)
#because numbers are very large,so instead of writing 1000000,2000000,3000000; it shows 1e6 scale


In [ ]:
sns.histplot(df[df['trip_distance'] < 50]['trip_distance'], bins=50)
plt.title("Trip Distance Distribution ")
plt.show()

In [ ]:
# Bar Plot(Payment Type Distribution)

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(x=df['payment_type'])

plt.title("Payment Type Distribution ( in  10 Lakhs )")
plt.xlabel("Payment Type")
plt.ylabel("Count (10 Lakhs)")

plt.show()

In [ ]:
# Fare Amount  Vs No.of rides(Histogram  Plot)

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    df[(df['fare_amount'] > 0) & (df['fare_amount'] < 200)]['fare_amount'],
    bins=40,

)

plt.title("Fare Amount vs Number of Rides")
plt.xlabel("Fare Amount")
plt.ylabel("Number of Rides")
plt.show()

In [ ]:
# Passenger Count Distribution(Count Plot)

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(x=df['passenger_count'])

plt.title("Passenger Count Distribution")
plt.show()

In [ ]:
# Bar Plot(Passenger Count vs Fare)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x='passenger_count', y='fare_amount', data=df)
plt.title("Passenger Count vs Fare")
plt.show()

In [ ]:
# Pickup Hour vs No. of rides(Box Plot)

In [ ]:
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour

sns.countplot(x='pickup_hour', data=df)
plt.title("Rides by Hour")
plt.show()

In [ ]:
# Outliers Detection (Fare Amount)

In [ ]:
Q1 = df['fare_amount'].quantile(0.25)
Q3 = df['fare_amount'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['fare_amount'] < lower) | (df['fare_amount'] > upper)]

print("Number of outliers:", len(outliers))

In [ ]:
# Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

features = ['fare_amount', 'trip_distance', 'passenger_count']

df[features] = scaler.fit_transform(df[features])

In [ ]:
df.head()
df[['fare_amount', 'trip_distance', 'passenger_count']].head()
df[features].describe()
print(df[features].head())

In [ ]:
features = ['fare_amount', 'trip_distance', 'passenger_count']

X = df[features]

In [ ]:
X = X.dropna() # handles missing values

## train the model

from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

model.fit(X_scaled)



In [ ]:
# used for making predictions

clean_df = df[features].dropna()


from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(clean_df)


from sklearn.ensemble import IsolationForest
model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
model.fit(X_scaled)


clean_df['anomaly'] = model.predict(X_scaled)

In [ ]:
clean_df['anomaly'].value_counts()

In [ ]:
clean_df['anomaly'] = clean_df['anomaly'].map({1: 0, -1: 1})

In [ ]:
clean_df['anomaly'].value_counts()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.scatter(clean_df['trip_distance'], clean_df['fare_amount'],
            c=clean_df['anomaly'], cmap='coolwarm', alpha=0.5)

plt.xlabel("Trip Distance")
plt.ylabel("Fare Amount")
plt.title("Anomaly Detection (Isolation Forest)")

plt.show()

In [ ]:
anomaly_percent = clean_df['anomaly'].mean() * 100
print(f"Anomaly Percentage: {anomaly_percent:.2f}%")

In [ ]:
clean_df[clean_df['anomaly'] == 1].describe()

In [ ]:
clean_df.groupby('anomaly')[['fare_amount','trip_distance']].mean()

In [ ]:
clean_df[clean_df['anomaly'] == 1].head(10)

In [ ]:
clean_df.to_parquet("/Users/riturajbhattacharjee/Desktop/yellow_tripdata_2023-02 (1).parquet", index=False)

In [ ]:
df_check = pd.read_parquet("/Users/riturajbhattacharjee/Desktop/yellow_tripdata_2023-02 (1).parquet")
df_check.head()

In [ ]:
clean_df[clean_df['anomaly'] == 1][['fare_amount','trip_distance','passenger_count']].describe()

In [ ]:
import pandas as pd
df = pd.read_parquet("/Users/riturajbhattacharjee/Desktop/yellow_tripdata_2023-02 (1).parquet")


In [ ]:
features = ['fare_amount', 'trip_distance', 'passenger_count']

X = df[features]

In [ ]:
X = X.dropna()

In [ ]:
features = ['fare_amount', 'trip_distance', 'passenger_count']
X = df[features].dropna()

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
model.fit(X_scaled)

In [ ]:
df['anomaly'] = model.predict(X_scaled)

In [ ]:
df['anomaly'] = df['anomaly'].map({1: 0, -1: 1})

In [ ]:
df['anomaly'].value_counts()

In [ ]:
df[df['anomaly'] == 1].head()

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df['trip_distance'], df['fare_amount'], c=df['anomaly'])
plt.xlabel("Trip Distance")
plt.ylabel("Fare Amount")
plt.title("Anomaly Detection")
plt.show()

### Anomaly Detection Results Overview

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# Reload the DataFrame to ensure 'df' is defined
df = pd.read_parquet("/content/yellow_tripdata_2023-02 (1).parquet")

# Define features as done previously
features = ['fare_amount', 'trip_distance', 'passenger_count']

# Prepare data for IsolationForest (as done in cell baa4091b-9c15-4af3-a6aa-d97225452e23)
X = df[features].dropna() # Ensure X is created from the reloaded df
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Re-train IsolationForest model and predict anomalies (as done in cell 857c4e09-bfe0-4463-aeb4-eed244224654 and e5e2f487-6956-4225-8531-7e1ba6f960ab)
model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
model.fit(X_scaled)
df['anomaly'] = model.predict(X_scaled)

# Map anomaly values (as done in cell 810d936d-14cc-4e0c-bb15-68a1c65e5d1a)
df['anomaly'] = df['anomaly'].map({1: 0, -1: 1})


print('Anomaly Counts:')
print(df['anomaly'].value_counts())

anomaly_percent = df['anomaly'].mean() * 100
print(f"\nAnomaly Percentage: {anomaly_percent:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print(classification_report(y_true, y_pred))